# Merge VitalDB Clinical + Lab Data for the 482 Arrhythmia-Annotated Cases

1. Extract the 482 `case_id`s from local `Annotation_file_*.csv` filenames
2. Filter `df_cases` (one row per case) and `df_labs` (long format) to those ids
3. Pivot labs to wide format (first draw per lab per case) and merge with clinical data
4. Report missingness per column
5. Save the merged table to `merged_482_cases.csv`

In [8]:
import re
from pathlib import Path
import pandas as pd

In [ ]:
BASE_DIR = Path("..").resolve()
ANNOTATION_DIR = BASE_DIR / "external_data" / "vitaldb-arrhythmia-database-1.0.0" / "Annotation_Files"
OUTPUT_CSV = BASE_DIR / "data" / "interim" / "merged_482_cases.csv"

## 1. Extract case_ids from annotation filenames

In [10]:
pattern = re.compile(r"Annotation_file_(\d+)\.csv$")
case_ids = sorted(
    int(m.group(1))
    for f in ANNOTATION_DIR.glob("Annotation_file_*.csv")
    if (m := pattern.match(f.name))
)

print(f"Found {len(case_ids)} annotation files")
assert len(case_ids) == len(set(case_ids)), "duplicate case_ids found"
print("First 5:", case_ids[:5])

Found 482 annotation files
First 5: [12, 13, 19, 42, 96]


## 2. Load and filter VitalDB clinical + lab data

In [11]:
df_cases = pd.read_csv("https://api.vitaldb.net/cases")
df_labs = pd.read_csv("https://api.vitaldb.net/labs")

print(f"df_cases: {df_cases.shape}, df_labs: {df_labs.shape}")

df_cases: (6388, 74), df_labs: (928448, 4)


In [12]:
cases_subset = df_cases[df_cases["caseid"].isin(case_ids)].copy()
labs_subset = df_labs[df_labs["caseid"].isin(case_ids)].copy()

print(f"cases_subset: {cases_subset.shape[0]} / {len(case_ids)} case_ids matched")

missing_from_cases = set(case_ids) - set(cases_subset["caseid"])
if missing_from_cases:
    print(f"WARNING: {len(missing_from_cases)} case_ids have no entry in df_cases: {sorted(missing_from_cases)}")

cases_subset: 482 / 482 case_ids matched


## 3. Pivot labs to wide format (first draw per lab per case)

In [13]:
# Keep the earliest (lowest dt) draw of each lab per case, then pivot long -> wide
labs_first = (
    labs_subset.sort_values("dt")
    .groupby(["caseid", "name"], as_index=False)
    .first()
)

labs_wide = labs_first.pivot(index="caseid", columns="name", values="result")
labs_wide.columns = [f"lab_{c}" for c in labs_wide.columns]
labs_wide = labs_wide.reset_index()

print(f"labs_wide: {labs_wide.shape} (one row per case, one column per lab)")

labs_wide: (458, 35) (one row per case, one column per lab)


## 4. Merge clinical + lab data into one row per patient

In [14]:
merged = cases_subset.merge(labs_wide, on="caseid", how="left")

assert merged["caseid"].is_unique, "merge produced duplicate rows per case"
print(f"merged: {merged.shape[0]} patients x {merged.shape[1]} columns")

merged: 482 patients x 108 columns


## 5. Missingness report

In [15]:
missingness = pd.DataFrame({
    "n_missing": merged.isna().sum(),
    "pct_missing": (merged.isna().mean() * 100).round(1),
}).sort_values("pct_missing", ascending=False)

pd.set_option("display.max_rows", None)
print(missingness.to_string())

                     n_missing  pct_missing
lmasize                    475         98.5
cline2                     465         96.5
lab_ammo                   453         94.0
aline2                     446         92.5
lab_ccr                    445         92.3
preop_be                   422         87.6
preop_sao2                 421         87.3
preop_paco2                421         87.3
preop_pao2                 421         87.3
preop_hco3                 421         87.3
preop_ph                   421         87.3
lab_esr                    404         83.8
dltubesize                 386         80.1
iv2                        363         75.3
lab_be                     340         70.5
cline1                     297         61.6
lab_p                      245         50.8
intraop_ebl                150         31.1
intraop_uo                 138         28.6
aline1                     132         27.4
lab_pco2                   124         25.7
lab_hco3                   124  

## 6. Save merged dataframe

In [16]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {merged.shape[0]} rows x {merged.shape[1]} columns to {OUTPUT_CSV.resolve()}")

Saved 482 rows x 108 columns to C:\Users\sukka\Downloads\MIMIC-IV\VitalDB Code\merged_482_cases.csv
